# Kepler exoplanet classification — Notebook 03

**Deep learning models: MLP, ResNet, and FT-Transformer**

Author: Atilla Ahmed

---

## Abstract

This notebook implements and evaluates three modern deep learning architectures for tabular data on the Kepler classification task: a Multi-Layer Perceptron (MLP), a residual-connection tabular network (ResNet-Tabular), and a Feature Tokenizer + Transformer (FT-Transformer).
Models are trained primarily on the leak-free feature configuration defined in Notebook 01 and benchmarked against the XGBoost baseline established in Notebook 02. A soft-voting ensemble combines the three architectures for the final result.

## Table of contents

1. [Data loading and training setup](#1-data-loading-and-training-setup)
2. [Multi-Layer Perceptron](#2-multi-layer-perceptron)
3. [Residual Tabular Network](#3-residual-tabular-network)
4. [Feature Tokenizer Transformer](#4-feature-tokenizer-transformer)
5. [Model ensemble](#5-model-ensemble)
6. [Summary](#6-summary)

## 1. Data loading and training setup

We load the processed splits from Notebook 01, prepare PyTorch tensors on the GPU, and define a single training function that will be used consistently for all three deep learning models.

### 1.1. Imports and configuration

In [1]:
import json
import warnings
from pathlib import Path
from typing import Callable

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    confusion_matrix,
    classification_report,
)

warnings.filterwarnings("ignore", category=FutureWarning)

RANDOM_SEED = 42
BATCH_SIZE = 256
MAX_EPOCHS = 100
PATIENCE = 15
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-5

PROCESSED_PATH = Path("../data/processed")
MODELS_PATH = Path("../models/dl")
MODELS_PATH.mkdir(parents=True, exist_ok=True)

int_to_class = {0: "CONFIRMED", 1: "CANDIDATE", 2: "FALSE POSITIVE"}
class_order = ["CONFIRMED", "CANDIDATE", "FALSE POSITIVE"]

torch.manual_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_SEED)

print("Setup complete")

Setup complete


### 1.2. GPU configuration

We detect the available compute device and prepare a global `device` variable used consistently across all model definitions and training loops.

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Device: {device}")
if device.type == "cuda":
    print(f"GPU:    {torch.cuda.get_device_name(0)}")

Device: cuda
GPU:    NVIDIA GeForce RTX 4050 Laptop GPU


### 1.3. Load processed data

We read the three parquet splits, load the feature-set definitions, and apply the same `StandardScaler` fit on training data used in Notebook 02. Since PyTorch models cannot participate in an `sklearn.pipeline.Pipeline` cleanly, we scale the features explicitly before creating tensors.

In [4]:
train_df = pd.read_parquet(PROCESSED_PATH / "train.parquet")
val_df = pd.read_parquet(PROCESSED_PATH / "val.parquet")
test_df = pd.read_parquet(PROCESSED_PATH / "test.parquet")

with open(PROCESSED_PATH / "feature_sets.json", "r") as f:
    feature_sets = json.load(f)

def split_features_target(df: pd.DataFrame) -> tuple[pd.DataFrame, pd.Series]:
    return df.drop(columns=["target"]), df["target"]

X_train, y_train = split_features_target(train_df)
X_val, y_val = split_features_target(val_df)
X_test, y_test = split_features_target(test_df)

feature_sets_effective = {
    "setup_leaky": [c for c in feature_sets["setup_a_leaky"] if c in X_train.columns],
    "setup_leak_free": [c for c in feature_sets["setup_c_leak_free"] if c in X_train.columns],
}

print(f"Train: {X_train.shape}")
print(f"Validation: {X_val.shape}")
print(f"Test: {X_test.shape}")
for name, features in feature_sets_effective.items():
    print(f" {name}: {len(features)} features")

Train: (6694, 101)
Validation: (1435, 101)
Test: (1435, 101)
 setup_leaky: 55 features
 setup_leak_free: 51 features


### 1.4. Prepare tensors and DataLoaders

We scale the features with `StandardScaler` fit on the training set only, convert everything to PyTorch tensors on the GPU, and wrap them in `DataLoader` objects for batched training.

In [5]:
def prepare_dataloaders(
    features: list[str],
    X_train: pd.DataFrame,
    y_train: pd.Series,
    X_val: pd.DataFrame,
    y_val: pd.Series,
    X_test: pd.DataFrame,
    y_test: pd.Series,
    batch_size: int = BATCH_SIZE,
) -> tuple[DataLoader, DataLoader, DataLoader, StandardScaler]:
    """Scale features and build train/val/test DataLoaders.

    Fits StandardScaler on train only, applies to val and test.
    Returns dataloaders and the fitted scaler for future use.
    """
    scaler = StandardScaler()
    X_tr = scaler.fit_transform(X_train[features])
    X_va = scaler.transform(X_val[features])
    X_te = scaler.transform(X_test[features])

    train_ds = TensorDataset(
        torch.tensor(X_tr, dtype=torch.float32),
        torch.tensor(y_train.values, dtype=torch.long),
    )
    val_ds = TensorDataset(
        torch.tensor(X_va, dtype=torch.float32),
        torch.tensor(y_val.values, dtype=torch.long),
    )
    test_ds = TensorDataset(
        torch.tensor(X_te, dtype=torch.float32),
        torch.tensor(y_test.values, dtype=torch.long),
    )

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    val_loader   = DataLoader(val_ds,   batch_size=batch_size, shuffle=False)
    test_loader  = DataLoader(test_ds,  batch_size=batch_size, shuffle=False)

    return train_loader, val_loader, test_loader, scaler


features_c = feature_sets_effective["setup_leak_free"]

train_loader, val_loader, test_loader, scaler_c = prepare_dataloaders(
    features_c, X_train, y_train, X_val, y_val, X_test, y_test
)

print(f"Features: {len(features_c)}")
print(f"Train batches:      {len(train_loader)}")
print(f"Validation batches: {len(val_loader)}")
print(f"Test batches:       {len(test_loader)}")

Features: 51
Train batches:      27
Validation batches: 6
Test batches:       6


### 1.5. Common training utility

We define a single training function used consistently across all three deep learning architectures. It implements the standard PyTorch training loop with AdamW optimizer, cosine learning-rate schedule, class-weighted cross-entropy loss, and early stopping on validation macro-F1.

In [6]:
def train_model(
    model: nn.Module,
    train_loader: DataLoader,
    val_loader: DataLoader,
    max_epochs: int = MAX_EPOCHS,
    patience: int = PATIENCE,
    learning_rate: float = LEARNING_RATE,
    weight_decay: float = WEIGHT_DECAY,
    class_weights: torch.Tensor = None,
    verbose: bool = True,
) -> tuple[nn.Module, dict]:
    """Train a PyTorch model with early stopping on validation macro-F1.

    Returns the trained model (with best weights restored) and a training history dict.
    """
    model = model.to(device)

    if class_weights is not None:
        class_weights = class_weights.to(device)
    criterion = nn.CrossEntropyLoss(weight=class_weights)

    optimizer = AdamW(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
    scheduler = CosineAnnealingLR(optimizer, T_max=max_epochs)

    history = {"train_loss": [], "val_loss": [], "val_macro_f1": []}
    best_f1 = -1.0
    best_state = None
    epochs_no_improve = 0

    for epoch in range(1, max_epochs + 1):
        model.train()
        train_losses = []
        for X_batch, y_batch in train_loader:
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)

            optimizer.zero_grad()
            logits = model(X_batch)
            loss = criterion(logits, y_batch)
            loss.backward()
            optimizer.step()

            train_losses.append(loss.item())

        scheduler.step()

        model.eval()
        val_losses = []
        val_preds = []
        val_targets = []
        with torch.no_grad():
            for X_batch, y_batch in val_loader:
                X_batch = X_batch.to(device)
                y_batch = y_batch.to(device)
                logits = model(X_batch)
                loss = criterion(logits, y_batch)
                val_losses.append(loss.item())
                val_preds.extend(logits.argmax(dim=1).cpu().numpy())
                val_targets.extend(y_batch.cpu().numpy())

        train_loss = np.mean(train_losses)
        val_loss = np.mean(val_losses)
        val_f1 = f1_score(val_targets, val_preds, average="macro")

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["val_macro_f1"].append(val_f1)

        if val_f1 > best_f1:
            best_f1 = val_f1
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1

        if verbose and (epoch % 5 == 0 or epoch == 1):
            print(f"Epoch {epoch:3d}: train_loss={train_loss:.4f}  val_loss={val_loss:.4f}  val_macro_f1={val_f1:.4f}")

        if epochs_no_improve >= patience:
            if verbose:
                print(f"Early stopping at epoch {epoch} (best macro-F1 = {best_f1:.4f})")
            break

    model.load_state_dict(best_state)
    return model, history

### 1.6. Common evaluation utility

We define an evaluation function that runs a trained model on a DataLoader and returns predictions, probabilities, and metric summaries. All models use the same protocol to ensure fair comparison.

In [7]:
def evaluate_model(
    model: nn.Module,
    loader: DataLoader,
) -> dict:
    """Evaluate a trained PyTorch model and return predictions plus metrics."""
    model.eval()

    all_preds = []
    all_probs = []
    all_targets = []

    with torch.no_grad():
        for X_batch, y_batch in loader:
            X_batch = X_batch.to(device)
            logits = model(X_batch)
            probs = F.softmax(logits, dim=1)
            preds = logits.argmax(dim=1)

            all_preds.extend(preds.cpu().numpy())
            all_probs.append(probs.cpu().numpy())
            all_targets.extend(y_batch.numpy())

    all_preds = np.array(all_preds)
    all_probs = np.vstack(all_probs)
    all_targets = np.array(all_targets)

    per_class_f1 = f1_score(all_targets, all_preds, average=None)

    return {
        "predictions": all_preds,
        "probabilities": all_probs,
        "targets": all_targets,
        "accuracy": accuracy_score(all_targets, all_preds),
        "macro_f1": f1_score(all_targets, all_preds, average="macro"),
        "f1_confirmed": per_class_f1[0],
        "f1_candidate": per_class_f1[1],
        "f1_false_pos": per_class_f1[2],
        "confusion_matrix": confusion_matrix(all_targets, all_preds),
    }